# 화면 경계의 납작한 후보를 제외하면 실제 원판도 사라질까?
기존 Mac 원판 모델의 빈 출력을 CutMix 모델로 보완한다. 검증 6명의 동작 중 미탐은 필터 전후 모두 5장이었다. b06의 빈 배경 오탐 84장을 제거했지만 b02에서는 화면 위로 잘린 실제 원판 2장을 새로 놓쳤다.

## 1. 원본부터 같은 조건으로 비교하기
새로 추론하려면 아래 값을 True로 바꾼다. 두 모델이 각각 첫 프레임부터 낸 후보를 필터 전후에 똑같이 사용한다. 원본 경계 1px 이내이면서 너비가 높이의 3배 이상인 후보만 시간 연결 **전에** 뺀다. 영상·모델·조건마다 연결 이력을 새로 시작한다. 수동 정답은 선택을 끝낸 후 채점에 쓴다.

In [ ]:
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "wlcv/config.py").is_file())
sys.path.insert(0, str(root))
import pandas as pd
from matplotlib import pyplot as plt
from wlcv.evaluation.plate_edge_filter import OUTPUT, infer_originals, replay, summarize, changed_outputs
from wlcv.evaluation.edge_candidate_review import candidate_sheets, plot_frame, plot_decisions

rerun_inference = False
if rerun_inference:
    infer_originals()
    pair = replay()
    candidate_sheets()
else:
    pair = pd.read_csv(OUTPUT / "paired.csv")
motion, backgrounds = summarize(pair)
changed = changed_outputs(pair)

## 2. 같은 정탐을 유지했는지 확인하기
정탐은 시간 연결 후 최종 박스의 IoU가 0.50 이상인 프레임이다. 손실과 복구를 따로 계산한다. a03의 기존 복구 8장은 두 조건을 같은 프레임 번호로 비교한다.

In [ ]:
display(motion)
points = pd.read_csv(OUTPUT / "a03_recoveries.csv")
display(points[["frame_index","source_before","source_after","iou_before","iou_after"]].round(3))

## 3. 빈 배경의 오탐과 연속 길이
b05와 b06은 학습에서 본 배경이다. 불이 켜진 칸 하나는 해당 프레임에서 원판 박스가 한 개 선택됐다는 의미다.

In [ ]:
display(backgrounds[backgrounds.method.eq("Past link")])
b06 = pair[(pair.video_id == "b06_sn_f_01") & (pair.method == "Past link")].sort_values("frame_index")
fig, ax = plt.subplots(figsize=(9,2.3), layout="constrained")
for y, condition in enumerate(["after","before"]):
    present = b06[f"x1_{condition}"].notna()
    ax.bar(b06.frame_index[present], [0.54]*sum(present), bottom=y-0.27, width=1, color="#007d81")
ax.set(xlabel="b06 frame", xlim=(-0.5,93.5), ylim=(-0.5,1.5), yticks=[0,1],
       yticklabels=["Filter on","Filter off"], title="False-positive frames on the empty-bar video")
fig.savefig(OUTPUT / "b06_timeline.png", dpi=150)
plt.show()

## 4. 화면 밖으로 잘린 실제 원판과 수동 정답
초록 박스는 수동 정답, 주황 박스는 각 조건의 최종 출력이다. b02의 마지막 수동 이벤트는 36번이다. 77·78번은 그 이후이므로 별도로 집계한다. 79–91번에는 수동 박스가 없어 출력 변화만 확인한다.

In [ ]:
val_changes = changed[(changed.method == "Past link") & changed.video_id.str.startswith("b02")]
display(val_changes[["frame_index","in_motion","source_before","source_after","iou_before","iou_after"]].round(3))
fig = plot_decisions(pair, [("b06_sn_f_01",0), ("b02_sn_h_02",77), ("b02_sn_h_02",78)])
fig.savefig(OUTPUT / "original_decisions.png", dpi=135)
plt.show()

## 5. 제외된 후보를 원본에서 직접 보기
주황은 Mac 베이스라인, 파랑은 CutMix 후보이다. 프레임 번호와 후보 순서를 바꿔 확대한다. 모든 제외 후보의 영상·프레임·페이지·칸은 로컬 candidate_sheets/index.csv에 있다.

In [ ]:
rejected = pd.read_csv(OUTPUT / "rejected.csv")
display(rejected.groupby("video_id").agg(제외후보=("model","size"), 해당프레임=("frame_index","nunique")))
video, frame = "b02_sn_h_02", 77
fig = plot_frame(video, frame, index_in_frame=0)
plt.show()

두 영상의 반례를 함께 보존한다. 천장 가장자리의 오탐은 사라졌고, 머리 위 원판의 보이는 일부분도 동일한 모양 조건을 만족했다. 새 촬영 환경에 일괄 적용할 근거는 부족하다.